# 02 — Models & Providers

**Tooling** — choose the engine. Swapping the model doesn't change what you ask the agent to do, only who
does the work and what it costs.

When you finish you can:

- Write a model string for any provider, and know which ones need an extra install
- Pass settings per run or per agent, including the provider-neutral `thinking`
- Keep an agent answering when a model fails, with `FallbackModel`
- Compare what two models charge for the same answer

In [ ]:
from learningdantic import setup

MODEL = setup()

## Model strings

The first argument of `Agent` is a string with a **prefix**. The prefix picks the provider and the API it speaks:

| Prefix | Meaning |
|---|---|
| `openai:` | OpenAI's **Responses API** (the modern default: reasoning and tools work together) |
| `openai-chat:` | OpenAI's older Chat Completions API |
| `openai-codex:` | your ChatGPT/Codex **subscription** instead of an API key (install the Codex CLI, then run `codex login`) |
| `anthropic:` | Claude models |
| `google:` | Gemini models |
| `xai:` | xAI/Grok models. Needs an extra install first: `uv add "pydantic-ai[xai]"` |

Support for `openai`, `anthropic` and `google` comes installed. A name without a prefix, like
`Agent('gpt-6.1-sol')`, raises `UserError`. If you hit that, you copied code from an old tutorial. (The error
suggests `openai-chat:`; this course uses `openai:`.)

In [ ]:
from pydantic_ai import Agent

agent = Agent(MODEL, instructions="Answer with one short sentence.")

result = await agent.run("In one sentence: why do agent frameworks exist at all?")
print(result.output)

> ✅ **Checkpoint** — a one-sentence answer from your `MODEL`. To try another provider, change `MODEL` in `.env`
> (for example to an `anthropic:` model, with `ANTHROPIC_API_KEY` set), restart the kernel and re-run.
> This cell doesn't change.

## Model settings

Settings are a plain dict. (`ModelSettings` is a `TypedDict`: a dict with a fixed set of known keys.)
Pass them **per run** with `agent.run(..., model_settings=...)`, or fix them **on the agent** with
`Agent(..., model_settings=...)`.

The setting to know first is `thinking`: how hard a reasoning model thinks before it answers.
`'low'` is fast and cheap, `'high'` is slow and thorough. (There are also `'minimal'`, `'medium'`, `'xhigh'`,
and `False` to turn thinking off where the model allows it.) It means the same thing for OpenAI, Anthropic,
Google and xAI, so it survives a `MODEL` swap. OpenAI's own spelling is `openai_reasoning_effort`; if you set
both, the provider's own setting wins.

What about `temperature`? Reasoning models don't use it. While reasoning is on, pydantic-ai drops it and
shows a `UserWarning`, so you'll know.

In [ ]:
# Per run: only this one call thinks less.
quick = await agent.run("In one word: is water wet?", model_settings={"thinking": "low"})
print("per run  :", quick.output)
# getattr(x, "name", 0) reads x.name, or gives 0 when x has no such attribute (not every model reports it).
print("reasoning tokens:", getattr(quick.usage, "output_reasoning_tokens", 0))

# Per agent: every run of this agent thinks less.
agent_fast = Agent(
    MODEL,
    instructions="Answer with one short sentence.",
    model_settings={"thinking": "low"},
)
r = await agent_fast.run("Why is the sky blue?")
print("per agent:", r.output)

> ✅ **Checkpoint** — two short answers (the one-word question may get just `Yes.`), and a small
> `reasoning tokens:` number: the thinking you paid for on that run (`output_reasoning_tokens`, from notebook 01; 0 if your model doesn't report it).
> Try `'high'` instead and that number usually grows. Nothing else in the code changes, only one dict: `thinking` sets how much effort (and money) goes into each answer, not how you call the agent.

## FallbackModel — when a model fails

Rate limits and outages happen. `FallbackModel` takes several models in order: if a request to the first one
fails with an API error, the same request goes to the next model. You still get one result.

To see it fire without waiting for a real outage, the first model below is a fake one that is always down.
`FunctionModel` (more in notebook 08) turns a plain function into a model. This function raises
`ModelHTTPError` with status 503, the "service unavailable" error a provider sends during an outage.
Your `MODEL` is the backup, so this cell works with any provider.

In [ ]:
from pydantic_ai import ModelHTTPError
from pydantic_ai.models.fallback import FallbackModel
from pydantic_ai.models.function import FunctionModel


def outage(messages, info):
    """A provider that is down: every request fails with HTTP 503."""
    raise ModelHTTPError(status_code=503, model_name="down-model", body="Service Unavailable")


resilient = Agent(
    # first choice (always down), then the backup
    FallbackModel(FunctionModel(outage, model_name="down-model"), MODEL),
    instructions="Say something reassuring about error handling. One sentence.",
)

r = await resilient.run("What happens if you fail?")
print(r.output)
print("answered by:", r.response.model_name)

> ✅ **Checkpoint** — a reassuring sentence, and `answered by:` names your `MODEL` (without its provider
> prefix), not `down-model`. The request to `down-model` failed with a 503, so the same request went to your
> `MODEL`, and no error reached you.
>
> Tip: in real code, put a model from a *different* provider second, so one provider's outage can't take out
> both.

## Compare models — the meter, side by side

Same prompt, several models: your `MODEL`, `openai:gpt-6.1-sol` and a Claude model. `result.usage` tells you
what each one charged. `has_key(model)` is `True` when that model's key (or login) is set up, and it sends
no request, so a model you can't reach prints `skipped: no key` instead of an error.

In [ ]:
from learningdantic import has_key

prompt = 'Explain what a "token" is, in exactly one sentence.'

models = {
    "MODEL": MODEL,
    "gpt-6.1-sol": "openai:gpt-6.1-sol",
    "claude-haiku-5-5": "anthropic:claude-haiku-5-5",
}

for label, model_string in models.items():
    # :16 pads the text to 16 characters so the columns line up ('' gives just the padding)
    if not has_key(model_string):
        print(f"{label:16} | skipped: no key")
        continue  # jump straight to the next model
    comparison_agent = Agent(model_string, instructions="Answer with exactly one sentence.")
    r = await comparison_agent.run(prompt)
    print(f"{label:16} | in={r.usage.input_tokens} out={r.usage.output_tokens} cost={r.usage.cost}")
    print(f"{'':16} | {r.output}")

> ✅ **Checkpoint** — one row per model you can reach (`skipped: no key` for the rest), with similar
> sentences and very different price tags. In our test run the `gpt-6.1-sol` row cost more than ten times the `MODEL` row for the
> same question. (If your `MODEL` already *is* `gpt-6.1-sol`, both rows cost about the same.) `cost=None` means the price is unknown, not free.
>
> 🔭 **In Logfire** (optional): the **LLMs view** does this comparison for you: calls, errors, latency,
> tokens per second and cost per model, across all your notebook runs.

## 🏋️ Exercise — the model shoot-out

Pick one prompt and run it through every model you can reach: your `MODEL`, `openai:gpt-6.1-sol`, and an
`anthropic:` or `google:` model if you have that key.

1. Collect `input_tokens`, `output_tokens` and `cost` for each model into a list.
2. Print the cheapest model. Careful: a `cost` of `None` means "price unknown", not "free", so leave those out.

Stretch: read the answers side by side. Was the quality difference worth the price?

Solution: [`../solutions/02_compare_models.ipynb`](../solutions/02_compare_models.ipynb).

## Next →

**[03_structured_outputs.ipynb](03_structured_outputs.ipynb)** — the feature that makes pydantic-ai *pydantic*:
validated Python objects instead of strings.